# Langchain agent

In [1]:
#This file contains code snippets for creating and using an agent with tools in LangChain.
# function vs tool
# langchain tool created for arthimetic operations
# agent created to use these tools based on user input

# Another scenario: Supply Chain Management Tool and Agent
# Another scenario: Financial Analysis Tool and Agent


In [2]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)


0.1.16
0.0.38


In [3]:
#!pip uninstall langchain -y && pip uninstall langchain-core -y && pip uninstall langchain-community -y
#!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24


In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Load environment variables
load_dotenv()

# Create model
model = ChatOpenAI(model="gpt-4o-mini")


/Users/goura/Desktop/Workspace/Agentic AI/py313/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [5]:
def add_numbers(a, b):
    return a + b

# Test it manually
print("Function Output:", add_numbers(5, 7))


Function Output: 12


# Convert It Into a Tool (LLM CAN Use It)

In [41]:
@tool
def add_numbers_tool(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b


from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent

# Agent Prompt Template
# prompt = ChatPromptTemplate.from_template("""
# You are a helpful assistant. Be concise and accurate..

# User Request: {input},
# ("placeholder", "{agent_scratchpad}"
# """)

prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful assistant. Be concise and accurate..

IMPORTANT:
- Use tools when required
- Once you get the final answer, exit the loop
- Do NOT call tools again if answer is already found
"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

tools = [add_numbers_tool]


In [43]:
# Create an agent using the LLM, tools, and prompt. 
# This defines how the agent should think and when to use tools, but does not execute anything yet.
agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)

# Wrap the agent in an executor so it can actually run, call tools, and generate outputs.
# 'verbose' shows step logs, and 'max_iterations' limits how many reasoning steps it can take.
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=10)
result = executor.invoke({"input": "Calculate the sum of 10 and 20. "})

print("Agent Output:", result)



> Entering new AgentExecutor chain...

Invoking: `add_numbers_tool` with `{'a': 10, 'b': 20}`


30The sum of 10 and 20 is 30.

> Finished chain.
Agent Output: {'input': 'Calculate the sum of 10 and 20. ', 'output': 'The sum of 10 and 20 is 30.'}


In [18]:
# test below:
# "what is the distance between earth and moon in km?"}]})


Try some basic tools

In [19]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x


In [ ]:
# modify prompt template for above tool
# new_prompt = ChatPromptTemplate.from_template("""
# You are an expert mathematician. Use the tools provided to solve arithmetic problems accurately.
# User Request: {input},
# ("placeholder", "{agent_scratchpad}"
# """)

new_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert mathematician.
This agent can take 4 tools - add, multiply, exponentiate, 
     

IMPORTANT RULES:
- Break the problem into steps
- Use tools step-by-step
- Do NOT stop after one calculation
- Continue until the full problem is solved
- Always use previous results for next steps
- if the output is greater than 5, don't invoke 'multiply' tool

"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [28]:
# create agent similar to above using these 4 tools
agent = create_tool_calling_agent(llm=model, tools=[add, multiply, exponentiate, subtract], prompt=new_prompt)
executor = AgentExecutor(agent=agent, tools=[add, multiply, exponentiate, subtract], verbose=True,max_iterations=3)


In [29]:
executor.invoke({"input": "What is the result of adding 5 and 3 and then multiply by 10"})




> Entering new AgentExecutor chain...

Invoking: `add` with `{'x': 5, 'y': 3}`


8.0
Invoking: `multiply` with `{'x': 8, 'y': 10}`


80.0The result of adding 5 and 3 is 8, and then multiplying that by 10 gives 80.

> Finished chain.


{'input': 'What is the result of adding 5 and 3 and then multiply by 10',
 'output': 'The result of adding 5 and 3 is 8, and then multiplying that by 10 gives 80.'}

In [35]:
new_output = executor.invoke({"input": "What is the result of adding 5 and 3, then multiplying by 2, and finally raising to the power of 2?"})# --- IGNORE ---
print("New Agent Output:", new_output)



> Entering new AgentExecutor chain...

Invoking: `add` with `{'x': 5, 'y': 3}`


8.0
Invoking: `multiply` with `{'x': 8, 'y': 2}`


16.0
Invoking: `add` with `{'x': 5, 'y': 3}`


8.0
Invoking: `add` with `{'x': 5, 'y': 3}`


8.0

> Finished chain.
New Agent Output: {'input': 'What is the result of adding 5 and 3, then multiplying by 2, and finally raising to the power of 2?', 'output': 'Agent stopped due to max iterations.'}


In [36]:
# test cases to try:
# "What is nine plus 10, minus 4 * 2, to the power of 3"
# "What is the result of 7 multiplied by 6, then subtracting 5, and finally raising to the power of 2?
# "calculate (8 + 2) * (5 - 3) to the power of 2"

# Scenario-1: Financial Analysis Tool and Agent

In [46]:
# industry_prompt = ChatPromptTemplate.from_template("""
# You are an expert business analyst. Use the tools provided to answer business-related queries accurately.
                                                   
# User Request: {input},
# ("placeholder", "{agent_scratchpad}"
# """)

industry_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert business analyst.

Instructions  
- Use tools when needed
- Avoid repeating the same tool call
- Once you have the final answer, respond clearly and stop
"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# industry specific tool
@tool
def get_financial_report(company_name: str, year: int) -> str:
    """Retrieve the financial report for a given company and year."""
    # Dummy data for illustration
    dummy_reports = {
        ("xyzInc", 2022): "XYZInc 2022 Financial Report: Revenue $10M, Profit $2M.",
        ("BizInc", 2022): "BizInc 2022 Financial Report: Revenue $5M, Profit $1M."
    }
    return dummy_reports.get((company_name, year), "Report not found.")
industry_tools = [get_financial_report]
industry_agent = create_tool_calling_agent(llm=model, tools=industry_tools, prompt=industry_prompt)
industry_executor = AgentExecutor(agent=industry_agent, tools=industry_tools, verbose=True,max_iterations=2)
industry_result = industry_executor.invoke({"input": "Get the financial report for xyzInc for the year 2022."})
print("Industry Agent Output:", industry_result)




> Entering new AgentExecutor chain...

Invoking: `get_financial_report` with `{'company_name': 'xyzInc', 'year': 2022}`


XYZInc 2022 Financial Report: Revenue $10M, Profit $2M.The financial report for XYZInc for the year 2022 shows a revenue of $10 million and a profit of $2 million.

> Finished chain.
Industry Agent Output: {'input': 'Get the financial report for xyzInc for the year 2022.', 'output': 'The financial report for XYZInc for the year 2022 shows a revenue of $10 million and a profit of $2 million.'}


# Scenario2: Supply Chain Management Tool and Agent

In [47]:
# sample_prompt = ChatPromptTemplate.from_template("""
# You are an expert supply chain analyst. Use the tools provided to answer supply chain-related queries accurately.
# User Request: {input},
# ("placeholder", "{agent_scratchpad}"
# """)

sample_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert supply chain analyst.

- Use tools when needed
- Avoid repeating the same tool call
- Once you have the final answer, respond clearly and stop
"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


dummy_inventory = {
        "item123": "In Stock: 150 units",
        "item456": "Out of Stock",
        "item789": "In Stock: 75 units",
        "item101": "Low Stock: 10 units",
    }
@tool
def get_inventory_status(item_id: str) -> str:
    """Retrieve the inventory status for a given item ID."""
    return dummy_inventory.get(item_id, "Item not found.")
supply_chain_tools = [get_inventory_status]
supply_chain_agent = create_tool_calling_agent(llm=model, tools=supply_chain_tools, prompt=sample_prompt)
supply_chain_executor = AgentExecutor(agent=supply_chain_agent, tools=supply_chain_tools, verbose=True,max_iterations=2)
supply_chain_result = supply_chain_executor.invoke({"input": "Get the inventory status for item ID item123."})
print("Supply Chain Agent Output:", supply_chain_result)        

# more test samples to try
# "Get the financial report for BizInc for the year 2022."
# "Get the inventory status for item ID item456."  




> Entering new AgentExecutor chain...

Invoking: `get_inventory_status` with `{'item_id': 'item123'}`


In Stock: 150 unitsThe inventory status for item ID item123 is in stock with 150 units available.

> Finished chain.
Supply Chain Agent Output: {'input': 'Get the inventory status for item ID item123.', 'output': 'The inventory status for item ID item123 is in stock with 150 units available.'}


# TO DO: Learners can try similar scenario for healthcare domain with tools like:
    - get_patient_record(patient_id: str) -> str
    - schedule_appointment(patient_id: str, date: str) -> str
    - get_medication_info(medication_name: str) -> str
